In [10]:
# Build the PFEP master table

import numpy as np
import pandas as pd

np.random.seed(42)  #You can use any number, seed produces the exact same random values. Without it, you'd get different data on every run.

parts = [f"P-{1000 + i}" for i in range(1, 21)]   # Getting 20 part Ids: P-1001, P1002, ...

# Creating DataFrame-pfea (Plan for Every Part with 20 rows)
pfep = pd.DataFrame({
    'part_id': parts,
    'description': [f'Component {i}' for i in range(1, 21)],
    'location': np.random.choice(['WH-A', 'WH-B', 'Line-1', 'Line-2'], 20),
    'avg_daily_demand': np.random.randint(20, 100, 20),    # avg units used per day
    'lead_time_days': np.random.randint(2, 10, 20)        # days for new stock to arrive once ordered
})


# avg_daily_demand = 50 (you use 50 units/day)
# lead_time_days = 4 (new stock takes 4 days to arrive after you order)
# 20% safety buffer
# reorder_point = 240 means: "When your stock drops to 240 units, order more NOW — because you'll burn through roughly 240 units in the time it takes the new order to arrive, and the buffer protects you if demand spikes."
pfep['reorder_point'] = (pfep['avg_daily_demand'] * pfep['lead_time_days'] * 1.2).astype(int)



# max_stock (full — start here, refill up to here)
# order enough to bring stock back up to max_stock
# mainly to set a sensible full-stock starting level per part, and as the reference for 'full'
pfep['max_stock'] = (pfep['reorder_point'] * 2).astype(int)

pfep.head()


,part_id,description,location,avg_daily_demand,lead_time_days,reorder_point,max_stock
0,P-1001,Component 1,Line-1,41,7,344,688
1,P-1002,Component 2,Line-2,72,8,691,1382
2,P-1003,Component 3,WH-A,21,7,176,352
3,P-1004,Component 4,Line-1,49,4,235,470
4,P-1005,Component 5,Line-1,57,5,342,684


In [11]:
# Simulate 30 days of stock movements

rows = []

# makes 30 consecutive dates (Jul 1-30). .date strips the time part
dates = pd.date_range('2026-07-01', periods=30).date

# iterrows holds (index, p) in a tuple
# _, p -> used for tuple unpacking
# programmers use _ by convention to signal "I'm required to catch this value, but I don't use it."
for _, p in pfep.iterrows():
  stock = p['max_stock']    #each part starts full as its max_stock
  for d in dates:
    # "I modeled daily consumption as a normal distribution around each part's average demand, so usage varies realistically day to day. I wrapped it in int() for whole units and max(0, …) so consumption can never go negative."
    consumed = max(0, int(np.random.normal(p['avg_daily_demand'], 10)))

    # "There's a 15% chance each day that a delivery arrives. If it does, it refills the part back up to its max_stock (received = the gap to full). Otherwise, received = 0 (nothing arrives)."
    # I kept the probability low on purpose so some parts drift below their reorder point, giving my alert layer real shortages to detect."
    received = (p['max_stock'] - stock) if np.random.rand() < 0.15 else 0
    opening = stock #stock at the start of the day
    closing = max(0, opening - consumed + received)   # max(0, ...) stops stock going negative
    rows.append([p['part_id'], d, opening, consumed, received, closing])
    stock = closing   # today's closing becomes tomorrow's opening

stock_daily = pd.DataFrame(rows, columns=['part_id', 'date', 'opening', 'consumed', 'received', 'closing'])   # Creating a DataFrame-stock_daily
print(stock_daily.shape)      # should be (600, 6) -> 20 parts × 30 days
stock_daily.head(10)


(600, 6)


,part_id,date,opening,consumed,received,closing
0,P-1001,2026-07-01,688,20,0,668
1,P-1001,2026-07-02,668,52,0,616
2,P-1001,2026-07-03,616,48,0,568
3,P-1001,2026-07-04,568,29,0,539
4,P-1001,2026-07-05,539,37,0,502
5,P-1001,2026-07-06,502,29,0,473
6,P-1001,2026-07-07,473,27,0,446
7,P-1001,2026-07-08,446,37,242,651
8,P-1001,2026-07-09,651,39,0,612
9,P-1001,2026-07-10,612,53,0,559


In [12]:
# Load both tables into the SQL warehouse

import sqlite3

# This opens a connection to a SQLite database — but instead of a file, it creates the database purely in RAM (memory). Rebuilt from data each run.
conn = sqlite3.connect(':memory:')

# conn — which database to write to (the connection from above).
pfep.to_sql('pfep', conn, if_exists='replace', index=False)
stock_daily.to_sql('stock_daily', conn, if_exists='replace', index=False)

# JOIN connects the two tables on their shared column, part_id. It brings "current stock" and "reorder threshold" onto the same row so you can compare them.
pd.read_sql('SELECT s.part_id, s.date, s.closing, p.reorder_point, p.location FROM stock_daily s JOIN pfep p ON s.part_id = p.part_id LIMIT 5', conn)

,part_id,date,closing,reorder_point,location
0,P-1001,2026-07-01,668,344,Line-1
1,P-1001,2026-07-02,616,344,Line-1
2,P-1001,2026-07-03,568,344,Line-1
3,P-1001,2026-07-04,539,344,Line-1
4,P-1001,2026-07-05,502,344,Line-1


In [13]:
# Compute the KPIs(current stock + days of coverage)


# 1. Get each part's LATEST day's closing stock (its current stock)
# The inner query (the part in parentheses, aliased m)
latest = pd.read_sql('SELECT s.part_id, s.closing AS current_stock FROM stock_daily s JOIN (SELECT part_id, MAX(date) AS last_date FROM stock_daily GROUP BY part_id) m ON s.part_id = m.part_id AND s.date = m.last_date', conn)

# 2. Join current stock to the PFEP master and compute coverage
kpi = latest.merge(pfep, on='part_id')

kpi['days_of_coverage'] = (kpi['current_stock'] / kpi['avg_daily_demand']).round(1)     # how many days until we run out ?    .round(1) keeps one decimal.

# shows the lowest coverage parts on the top
kpi[['part_id', 'location', 'current_stock', 'reorder_point', 'avg_daily_demand', 'days_of_coverage']].sort_values('days_of_coverage').head(10)


,part_id,location,current_stock,reorder_point,avg_daily_demand,days_of_coverage
2,P-1003,WH-A,0,176,21,0.0
14,P-1015,Line-2,0,187,78,0.0
18,P-1019,Line-2,0,81,34,0.0
19,P-1020,Line-1,0,486,81,0.0
9,P-1010,WH-B,168,249,52,3.2
8,P-1009,Line-1,148,96,40,3.7
3,P-1004,Line-1,230,235,49,4.7
11,P-1012,Line-1,381,369,77,4.9
16,P-1017,Line-2,421,284,79,5.3
6,P-1007,WH-A,536,498,83,6.5


In [14]:
# The alert layer
def stock_alerts(kpi):
  alerts = []

  for _, r in kpi.iterrows():
    if r['current_stock'] == 0:
      alerts.append(
          f"🔴 STOCKOUT: {r['part_id']} is OUT OF STOCK at {r['location']}")
    elif r['current_stock'] < r['reorder_point']:
      alerts.append(
          f"🟠 REORDER: {r['part_id']} at {r['current_stock']} units "
          f"reorder point {r['reorder_point']}, ~ {r['days_of_coverage']} days left")
    elif r['days_of_coverage'] < 4:
      alerts.append(
          f"🟡 LOW COVERAGE: {r['part_id']} has only {r['days_of_coverage']} days left")

  return alerts

alerts = stock_alerts(kpi)

print(f'{len(alerts)} alert(s) raised:\n')
for a in alerts:
    print(" ", a)

8 alert(s) raised:

  🔴 STOCKOUT: P-1003 is OUT OF STOCK at WH-A
  🟠 REORDER: P-1004 at 230 units reorder point 235, ~ 4.7 days left
  🟠 REORDER: P-1006 at 150 units reorder point 201, ~ 7.1 days left
  🟡 LOW COVERAGE: P-1009 has only 3.7 days left
  🟠 REORDER: P-1010 at 168 units reorder point 249, ~ 3.2 days left
  🔴 STOCKOUT: P-1015 is OUT OF STOCK at Line-2
  🔴 STOCKOUT: P-1019 is OUT OF STOCK at Line-2
  🔴 STOCKOUT: P-1020 is OUT OF STOCK at Line-1


In [15]:
stock_daily.head()

,part_id,date,opening,consumed,received,closing
0,P-1001,2026-07-01,688,20,0,668
1,P-1001,2026-07-02,668,52,0,616
2,P-1001,2026-07-03,616,48,0,568
3,P-1001,2026-07-04,568,29,0,539
4,P-1001,2026-07-05,539,37,0,502


In [16]:
# A data-quality check

# recompute what closing SHOULD be, and compare to what it IS
check = stock_daily.copy()

check['expected_closing'] = (check['opening'] - check['consumed'] + check['received']).clip(lower=0)    #.clip(lower=0) is the pandas way of doing max(0, ...) across a whole column

bad_rows = check[check['closing'] != check['expected_closing']]

print(f'Data-quality check: {len(bad_rows)} inconsistent row(s) out of {len(check)}')
if len(bad_rows) == 0:
  print("✅ All stock movements reconcile (closing = opening − consumed + received)")
else:
  print("❌ Found rows where closing does not reconcile:")
  print(bad_rows.head())

Data-quality check: 0 inconsistent row(s) out of 600
✅ All stock movements reconcile (closing = opening − consumed + received)
